In [ ]:
import chromadb
from chromadb.config import Settings
import json

class MyVectorDBConnector:
    def __init__(self, collection_name):
        chroma_client = chromadb.Client(Settings(allow_reset=True))
        self.collection = chroma_client.get_or_create_collection(name=collection_name)

In [ ]:
import os
from typing import List
from dotenv import load_dotenv
from openai import OpenAI

class MyVectorDBOpt:
    """向量数据库操作类（基于OpenAI Embeddings API）"""

    def __init__(self, api_key: str, base_url: str, default_model: str = "text-embedding-ada-002"):
        """
        初始化客户端
        :param api_key: API密钥
        :param base_url: API基础URL
        :param default_model: 默认使用的嵌入模型名称
        """
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.default_model = default_model

    def get_embeddings(self, texts: List[str], model: str = None) -> List[List[float]]:
        """
        获取文本的嵌入向量
        :param texts: 文本列表
        :param model: 嵌入模型名称，若不指定则使用默认模型
        :return: 嵌入向量列表
        """
        if not texts:
            return []
        model = model or self.default_model
        try:
            response = self.client.embeddings.create(input=texts, model=model)
            return [item.embedding for item in response.data]
        except Exception as e:
            raise RuntimeError(f"获取嵌入向量失败: {e}")

    def add_embeddings(self, embeddings: List[List[float]]) -> None:
        """
        将嵌入向量添加到向量数据库（待实现具体存储逻辑）
        :param embeddings: 嵌入向量列表
        """
        # TODO: 实现向量存储（例如插入到Milvus、Chroma或本地文件）
        raise NotImplementedError("add_embeddings 方法尚未实现")

    def search(self, query: str, top_k: int = 5) -> List[dict]:
        """
        基于查询文本执行向量检索（待实现）
        :param query: 查询文本
        :param top_k: 返回前k个最相似的结果
        :return: 检索结果列表
        """
        # TODO: 获取查询向量，然后与已存储的向量计算相似度并返回最相似的记录
        raise NotImplementedError("search 方法尚未实现")


# 使用示例（放在 if __name__ 中避免模块导入时执行）
if __name__ == "__main__":
    load_dotenv()  # 加载 .env 文件中的环境变量

    # 创建实例（注意保存变量，以便后续调用方法）
    vector_db = MyVectorDBOpt(
        api_key=os.getenv("BASE_EMBEDDINGS_API_KEY", ""),
        base_url=os.getenv("BASE_EMBEDDINGS_API_URL", ""),
        default_model="text-embedding-ada-002"  # 可在此修改默认模型
    )

    # 示例：获取嵌入向量
    sample_texts = ["Hello world", "How to optimize Python code"]
    try:
        embeddings = vector_db.get_embeddings(sample_texts)
        print(f"成功获取 {len(embeddings)} 个嵌入向量，每个向量的维度: {len(embeddings[0]) if embeddings else 0}")
    except Exception as e:
        print(e)